In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

!nvidia-smi

CUDA available: True
GPU count: 2
Sun Oct 18 11:32:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

In [3]:
import json
from pathlib import Path

checkpointpaths = []

for configurationpath in Path("/kaggle/input").rglob("config.json"):
    if "pa2sftcheckpoint" not in str(configurationpath).lower():
        continue

    candidatepath = configurationpath.parent

    hasweights = (
        (candidatepath / "model.safetensors").is_file()
        or (candidatepath / "model.safetensors.index.json").is_file()
    )

    if hasweights and (candidatepath / "tokenizer_config.json").is_file():
        checkpointpaths.append(candidatepath)

if len(checkpointpaths) != 1:
    raise RuntimeError(
        "Expected one SFT checkpoint under a pa2sftcheckpoint input, "
        f"found: {checkpointpaths}. Attach the private SFT checkpoint dataset."
    )

checkpointpath = checkpointpaths[0]

checkpointmetadata = json.loads(
    (checkpointpath / "checkpoint.json").read_text(encoding="utf-8")
)

print("Checkpoint path:", checkpointpath)
print(json.dumps(checkpointmetadata, indent=2))

Checkpoint path: /kaggle/input/datasets/kangerz22/pa2sftcheckpoint
{
  "optimizer_step": 938,
  "validation_loss": 1.3795008834900906,
  "selection": "lowest validation loss"
}


In [5]:
import os
import subprocess
import tempfile
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repositoryurl = "https://github.com/tachytach1/pa2-small-model-alignment.git"
repositorypath = Path("/kaggle/working/pa2-small-model-alignment")

if repositorypath.exists():
    print("Using existing repository:", repositorypath)
else:
    githubtoken = UserSecretsClient().get_secret("GITHUB_TOKEN")

    with tempfile.TemporaryDirectory() as temporarydirectory:
        helperpath = Path(temporarydirectory) / "git-askpass.sh"

        helperpath.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
            '  *Password*) printf "%s\\n" "$KAGGLE_GITHUB_TOKEN" ;;\n'
            'esac\n',
            encoding="utf-8",
        )
        helperpath.chmod(0o700)

        environment = os.environ.copy()
        environment["GIT_ASKPASS"] = str(helperpath)
        environment["GIT_TERMINAL_PROMPT"] = "0"
        environment["KAGGLE_GITHUB_TOKEN"] = githubtoken

        try:
            subprocess.run(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "clone",
                    repositoryurl,
                    str(repositorypath),
                ],
                env=environment,
                check=True,
            )
        finally:
            environment.pop("KAGGLE_GITHUB_TOKEN", None)
            del githubtoken

Cloning into '/kaggle/working/pa2-small-model-alignment'...


In [6]:
%cd /kaggle/working/pa2-small-model-alignment
!git log -1 --oneline

/kaggle/working/pa2-small-model-alignment
e2a74e0 (HEAD -> main, origin/main, origin/HEAD) Fix package configuration for alignment module


In [7]:
%%bash
set -e

python -m pip uninstall -y -q torchao
python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest matplotlib
python -m pip install -q -e . --no-deps
python -m pytest -q tests/test_public_adapter_smoke.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00
.......                                                                  [100%]
7 passed in 1.98s


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [8]:
%%writefile alignment/preferences.py
from __future__ import annotations

import hashlib
import json
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import HfApi
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from alignment.batching import encode_prompt_response, pad_encoded_examples
from alignment.data import formatinstruction


def extracthhpair(sourcerow):
    chosenconversation = sourcerow.get("chosen")
    rejectedconversation = sourcerow.get("rejected")

    if not isinstance(chosenconversation, str):
        return None

    if not isinstance(rejectedconversation, str):
        return None

    conversations = []

    for conversation in (chosenconversation, rejectedconversation):
        conversation = conversation.strip()

        if not conversation.startswith("Human:"):
            return None

        if conversation.count("\n\nHuman:") != 0:
            return None

        if conversation.count("\n\nAssistant:") != 1:
            return None

        instruction, response = conversation.split("\n\nAssistant:", 1)
        instruction = instruction.removeprefix("Human:").strip()
        response = response.strip()

        if not instruction or not response:
            return None

        conversations.append((instruction, response))

    choseninstruction, chosenresponse = conversations[0]
    rejectedinstruction, rejectedresponse = conversations[1]

    if choseninstruction != rejectedinstruction:
        return None

    return choseninstruction, chosenresponse, rejectedresponse


def extractultrafeedbackpair(sourcerow):
    chosenmessages = sourcerow.get("chosen")
    rejectedmessages = sourcerow.get("rejected")

    exchanges = []

    for messages in (chosenmessages, rejectedmessages):
        if not isinstance(messages, list) or len(messages) != 2:
            return None

        usermessage, assistantmessage = messages

        if not isinstance(usermessage, dict):
            return None

        if not isinstance(assistantmessage, dict):
            return None

        if usermessage.get("role") != "user":
            return None

        if assistantmessage.get("role") != "assistant":
            return None

        instruction = usermessage.get("content")
        response = assistantmessage.get("content")

        if not isinstance(instruction, str) or not isinstance(response, str):
            return None

        instruction = instruction.strip()
        response = response.strip()

        if not instruction or not response:
            return None

        exchanges.append((instruction, response))

    choseninstruction, chosenresponse = exchanges[0]
    rejectedinstruction, rejectedresponse = exchanges[1]

    if choseninstruction != rejectedinstruction:
        return None

    return choseninstruction, chosenresponse, rejectedresponse


def collatepreferencebatch(examples, paddingtokenid):
    chosenexamples = []
    rejectedexamples = []

    for example in examples:
        chosenexamples.append(example["chosen"])
        rejectedexamples.append(example["rejected"])

    return {
        "chosen": pad_encoded_examples(chosenexamples, paddingtokenid),
        "rejected": pad_encoded_examples(rejectedexamples, paddingtokenid),
    }


def preparepreferencesplit(
    tokenizer,
    datasetname,
    datasetrevision,
    sourcesplit,
    extractor,
    requestedpairs,
    maximumlength,
    maximumpromptlength,
    seed,
    shufflebuffer,
    seeninstructions,
):
    sourcedataset = load_dataset(
        datasetname,
        revision=datasetrevision,
        split=sourcesplit,
        streaming=True,
    ).shuffle(seed=seed, buffer_size=shufflebuffer)

    acceptedpairs = []

    statistics = {
        "source_split": sourcesplit,
        "examined_pairs": 0,
        "unsupported_or_invalid_pairs": 0,
        "duplicate_instructions": 0,
        "identical_response_pairs": 0,
        "long_prompt_pairs": 0,
        "pairs_without_response_tokens": 0,
        "identical_tokenized_pairs": 0,
        "accepted_pairs": 0,
        "chosen_response_tokens": 0,
        "rejected_response_tokens": 0,
    }

    with tqdm(
        total=requestedpairs,
        desc=f"Preparing {sourcesplit}",
        unit="pair",
    ) as progress:
        for sourcerow in sourcedataset:
            statistics["examined_pairs"] += 1
            extractedpair = extractor(sourcerow)

            if extractedpair is None:
                statistics["unsupported_or_invalid_pairs"] += 1
                continue

            instruction, chosenresponse, rejectedresponse = extractedpair

            instructionhash = hashlib.sha256(
                instruction.encode("utf-8")
            ).hexdigest()

            if instructionhash in seeninstructions:
                statistics["duplicate_instructions"] += 1
                continue

            if chosenresponse == rejectedresponse:
                statistics["identical_response_pairs"] += 1
                continue

            prompt = formatinstruction(instruction)
            promptlength = len(
                tokenizer.encode(prompt, add_special_tokens=False)
            )

            if promptlength > maximumpromptlength:
                statistics["long_prompt_pairs"] += 1
                continue

            chosenexample = encode_prompt_response(
                tokenizer,
                prompt,
                chosenresponse,
                maximumlength,
            )
            rejectedexample = encode_prompt_response(
                tokenizer,
                prompt,
                rejectedresponse,
                maximumlength,
            )

            chosenresponsecount = 0
            rejectedresponsecount = 0

            for tokenlabel in chosenexample["labels"][1:]:
                if tokenlabel != -100:
                    chosenresponsecount += 1

            for tokenlabel in rejectedexample["labels"][1:]:
                if tokenlabel != -100:
                    rejectedresponsecount += 1

            if chosenresponsecount == 0 or rejectedresponsecount == 0:
                statistics["pairs_without_response_tokens"] += 1
                continue

            if chosenexample["input_ids"] == rejectedexample["input_ids"]:
                statistics["identical_tokenized_pairs"] += 1
                continue

            acceptedpairs.append(
                {
                    "instruction_hash": instructionhash,
                    "instruction": instruction,
                    "prompt": prompt,
                    "chosen_response": chosenresponse,
                    "rejected_response": rejectedresponse,
                    "chosen": chosenexample,
                    "rejected": rejectedexample,
                    "chosen_response_token_count": chosenresponsecount,
                    "rejected_response_token_count": rejectedresponsecount,
                }
            )

            seeninstructions.add(instructionhash)
            statistics["accepted_pairs"] += 1
            statistics["chosen_response_tokens"] += chosenresponsecount
            statistics["rejected_response_tokens"] += rejectedresponsecount

            progress.update(1)

            if len(acceptedpairs) == requestedpairs:
                break

    if len(acceptedpairs) != requestedpairs:
        raise RuntimeError(
            f"{sourcesplit} supplied {len(acceptedpairs)} usable pairs, "
            f"but {requestedpairs} were requested. "
            f"Filtering statistics: {statistics}"
        )

    return Dataset.from_list(acceptedpairs), statistics


def preparepreferences(
    checkpointpath,
    track,
    outputpath,
    trainingpairs=15000,
    validationpairs=1000,
    maximumlength=512,
    maximumpromptlength=256,
    seed=42,
    shufflebuffer=1000,
    datasetrevision="main",
):
    for argumentname, argumentvalue in {
        "trainingpairs": trainingpairs,
        "validationpairs": validationpairs,
        "maximumlength": maximumlength,
        "maximumpromptlength": maximumpromptlength,
        "shufflebuffer": shufflebuffer,
    }.items():
        if argumentvalue < 1:
            raise ValueError(f"{argumentname} must be positive.")

    if maximumpromptlength >= maximumlength:
        raise ValueError("The prompt limit must leave room for response tokens.")

    if track == "hh":
        datasetname = "Anthropic/hh-rlhf"
        trainingsplit = "train"
        validationsplit = "test"
        extractor = extracthhpair
    elif track == "ultrafeedback":
        datasetname = "HuggingFaceH4/ultrafeedback_binarized"
        trainingsplit = "train_prefs"
        validationsplit = "test_prefs"
        extractor = extractultrafeedbackpair
    else:
        raise ValueError("track must be hh or ultrafeedback.")

    checkpointpath = Path(checkpointpath)
    destination = Path(outputpath)

    if destination.exists():
        raise FileExistsError(f"Output directory already exists: {destination}")

    tokenizer = AutoTokenizer.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        fix_mistral_regex=False,
    )

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise ValueError("The tokenizer needs a padding or EOS token.")
        tokenizer.pad_token = tokenizer.eos_token

    resolvedrevision = HfApi().dataset_info(
        datasetname,
        revision=datasetrevision,
    ).sha

    seeninstructions = set()

    trainingdataset, trainingstatistics = preparepreferencesplit(
        tokenizer,
        datasetname,
        resolvedrevision,
        trainingsplit,
        extractor,
        trainingpairs,
        maximumlength,
        maximumpromptlength,
        seed,
        shufflebuffer,
        seeninstructions,
    )

    validationdataset, validationstatistics = preparepreferencesplit(
        tokenizer,
        datasetname,
        resolvedrevision,
        validationsplit,
        extractor,
        validationpairs,
        maximumlength,
        maximumpromptlength,
        seed,
        shufflebuffer,
        seeninstructions,
    )

    manifest = {
        "track": track,
        "dataset_name": datasetname,
        "dataset_revision": resolvedrevision,
        "sft_checkpoint": str(checkpointpath),
        "checkpoint_metadata": json.loads(
            (checkpointpath / "checkpoint.json").read_text(encoding="utf-8")
        ),
        "tokenizer_file_sha256": hashlib.sha256(
            (checkpointpath / "tokenizer.json").read_bytes()
        ).hexdigest(),
        "seed": seed,
        "shuffle_buffer": shufflebuffer,
        "maximum_sequence_length": maximumlength,
        "maximum_prompt_length": maximumpromptlength,
        "long_prompt_policy": "reject; do not truncate the instruction separately",
        "conversation_policy": "single user and assistant exchange only",
        "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
        "padding_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "training": trainingstatistics,
        "validation": validationstatistics,
    }

    destination.mkdir(parents=True, exist_ok=False)

    DatasetDict(
        {
            "train": trainingdataset,
            "validation": validationdataset,
        }
    ).save_to_disk(str(destination / "dataset"))

    (destination / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(json.dumps(manifest, indent=2))
    print("Prepared preference data:", destination.resolve())

    return manifest

Writing alignment/preferences.py


In [12]:
import gc
import json
import math
import time
from functools import partial
from pathlib import Path

import torch

from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from alignment.dpo import computepreferencelossandmetrics
from alignment.preferences import collatepreferencebatch
from alignment.scoring import compute_batch_response_statistics


def collatecachedpreferences(examples, paddingtokenid):
    preferencebatch = collatepreferencebatch(
        examples,
        paddingtokenid,
    )

    chosenscores = []
    rejectedscores = []

    for example in examples:
        chosenscores.append(example["reference_chosen_logps"])
        rejectedscores.append(example["reference_rejected_logps"])

    preferencebatch["referencechosenscores"] = torch.tensor(
        chosenscores,
        dtype=torch.float32,
    )
    preferencebatch["referencerejectedscores"] = torch.tensor(
        rejectedscores,
        dtype=torch.float32,
    )

    return preferencebatch


def computecachedpreferenceloss(model, preferencebatch, beta):
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        chosenstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["chosen"],
        )
        rejectedstatistics = compute_batch_response_statistics(
            model,
            preferencebatch["rejected"],
        )

    chosenprobabilities = chosenstatistics[0]
    rejectedprobabilities = rejectedstatistics[0]

    loss, metrics = computepreferencelossandmetrics(
        chosenprobabilities,
        rejectedprobabilities,
        preferencebatch["referencechosenscores"],
        preferencebatch["referencerejectedscores"],
        beta,
    )

    device = chosenprobabilities.device

    chosendifference = (
        chosenprobabilities.detach()
        - preferencebatch["referencechosenscores"].to(device)
    ).abs().max()

    rejecteddifference = (
        rejectedprobabilities.detach()
        - preferencebatch["referencerejectedscores"].to(device)
    ).abs().max()

    metrics["maximum_reference_difference"] = torch.maximum(
        chosendifference,
        rejecteddifference,
    )

    return loss, metrics


def evaluatepreferences(model, loader, beta):
    previousmode = model.training
    model.eval()

    totals = {
        "dpo_loss": 0.0,
        "reward_accuracy": 0.0,
        "reward_margin": 0.0,
        "chosen_reward": 0.0,
        "rejected_reward": 0.0,
    }

    totalpairs = 0
    maximumdifference = 0.0

    try:
        with torch.no_grad():
            for preferencebatch in loader:
                loss, metrics = computecachedpreferenceloss(
                    model,
                    preferencebatch,
                    beta,
                )

                paircount = preferencebatch["referencechosenscores"].numel()
                totalpairs += paircount

                for metricname in totals:
                    totals[metricname] += (
                        float(metrics[metricname].item()) * paircount
                    )

                maximumdifference = max(
                    maximumdifference,
                    float(metrics["maximum_reference_difference"].item()),
                )
    finally:
        model.train(previousmode)

    if totalpairs == 0:
        raise ValueError("Validation contains no preference pairs.")

    results = {}

    for metricname, metrictotal in totals.items():
        results[metricname] = metrictotal / totalpairs

        if not math.isfinite(results[metricname]):
            raise FloatingPointError(
                f"Validation metric is not finite: {metricname}"
            )

    results["pairs"] = totalpairs
    results["maximum_reference_difference"] = maximumdifference

    return results


def appendpreferencemetrics(destination, record):
    with destination.open("a", encoding="utf-8") as outputfile:
        outputfile.write(
            json.dumps(record, allow_nan=False) + "\n"
        )


def savepreferencecheckpoint(model, tokenizer, destination, metadata):
    destination.mkdir(parents=True, exist_ok=True)

    previouscache = model.config.use_cache
    model.config.use_cache = True

    try:
        model.save_pretrained(
            str(destination),
            safe_serialization=True,
        )
        tokenizer.save_pretrained(str(destination))

        (destination / "checkpoint.json").write_text(
            json.dumps(metadata, indent=2, allow_nan=False),
            encoding="utf-8",
        )
    finally:
        model.config.use_cache = previouscache

In [13]:
def trainpreferences(
    cachepath,
    outputpath,
    accumulation=4,
    maxsteps=10,
    epochs=1,
    beta=0.1,
    learningrate=1e-6,
    evaluationinterval=5,
    seed=42,
):
    if accumulation < 1 or epochs < 1 or evaluationinterval < 1:
        raise ValueError("Training counts must be positive.")

    if maxsteps < 0:
        raise ValueError("maxsteps must be nonnegative.")

    if not math.isfinite(beta) or beta <= 0:
        raise ValueError("beta must be finite and positive.")

    if not math.isfinite(learningrate) or learningrate <= 0:
        raise ValueError("learningrate must be finite and positive.")

    cachepath = Path(cachepath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"Training output already exists: {outputpath}"
        )

    cachemanifest = json.loads(
        (cachepath / "manifest.json").read_text(encoding="utf-8")
    )

    if (
        cachemanifest["reference_checkpoint_metadata"]
        != checkpointmetadata
    ):
        raise ValueError("Reference cache belongs to a different checkpoint.")

    dataset = load_from_disk(str(cachepath / "dataset"))

    if len(dataset["train"]) == 0 or len(dataset["validation"]) == 0:
        raise ValueError("Training and validation datasets must not be empty.")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device = torch.device("cuda:0")
    torch.cuda.reset_peak_memory_stats(device)

    tokenizer = AutoTokenizer.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        fix_mistral_regex=False,
    )

    collator = partial(
        collatecachedpreferences,
        paddingtokenid=tokenizer.pad_token_id,
    )

    generator = torch.Generator().manual_seed(seed)

    trainingloader = DataLoader(
        dataset["train"],
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    validationloader = DataLoader(
        dataset["validation"],
        batch_size=1,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
    )

    plannedsteps = math.ceil(len(trainingloader) / accumulation) * epochs

    if maxsteps:
        plannedsteps = min(plannedsteps, maxsteps)

    print("Loading trainable SFT policy.", flush=True)

    model = AutoModelForCausalLM.from_pretrained(
        str(checkpointpath),
        local_files_only=True,
        dtype=torch.float32,
        attn_implementation="sdpa",
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learningrate,
        weight_decay=0.0,
        fused=True,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=2048.0,
        growth_interval=2000,
    )

    configuration = {
        "cache_manifest": cachemanifest,
        "policy_initialization": str(checkpointpath),
        "batch_size": 1,
        "gradient_accumulation": accumulation,
        "nominal_effective_batch_size": accumulation,
        "epochs": epochs,
        "planned_optimizer_attempts": plannedsteps,
        "beta": beta,
        "optimizer": "AdamW",
        "learning_rate": learningrate,
        "scheduler": "none",
        "weight_decay": 0.0,
        "gradient_clipping": 1.0,
        "gradient_checkpointing": True,
        "precision": "FP32 weights with FP16 autocast and GradScaler",
        "initial_gradient_scale": 2048.0,
        "scale_growth_interval": 2000,
        "seed": seed,
        "checkpoint_selection": "highest validation reward accuracy",
        "torch_version": torch.__version__,
    }

    outputpath.mkdir(parents=True, exist_ok=False)

    (outputpath / "configuration.json").write_text(
        json.dumps(configuration, indent=2),
        encoding="utf-8",
    )

    metricspath = outputpath / "metrics.jsonl"
    starttime = time.perf_counter()
    attemptedsteps = 0
    optimizersteps = 0
    skippedsteps = 0
    consecutiveskips = 0
    beststep = 0

    try:
        print("Evaluating the initial policy.", flush=True)

        validationresults = evaluatepreferences(
            model,
            validationloader,
            beta,
        )

        print(
            "Initial validation:",
            json.dumps(validationresults, indent=2),
            flush=True,
        )

        appendpreferencemetrics(
            metricspath,
            {
                "event": "validation",
                "optimizer_step": 0,
                **validationresults,
                "elapsed_seconds": time.perf_counter() - starttime,
            },
        )

        bestaccuracy = validationresults["reward_accuracy"]

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "best",
            {
                "optimizer_step": 0,
                "validation": validationresults,
            },
        )

        model.train()

        with tqdm(
            total=plannedsteps,
            desc="DPO optimizer attempts",
        ) as progress:
            for epoch in range(epochs):
                iterator = iter(trainingloader)

                while attemptedsteps < plannedsteps:
                    batches = []

                    for microbatchnumber in range(accumulation):
                        preferencebatch = next(iterator, None)

                        if preferencebatch is None:
                            break

                        batches.append(preferencebatch)

                    if not batches:
                        break

                    windowpairs = sum(
                        batch["referencechosenscores"].numel()
                        for batch in batches
                    )

                    optimizer.zero_grad(set_to_none=True)
                    windowloss = 0.0

                    for preferencebatch in batches:
                        loss, metrics = computecachedpreferenceloss(
                            model,
                            preferencebatch,
                            beta,
                        )

                        paircount = (
                            preferencebatch["referencechosenscores"].numel()
                        )

                        weightedloss = loss * paircount / windowpairs
                        scaler.scale(weightedloss).backward()

                        windowloss += (
                            float(loss.detach().item())
                            * paircount / windowpairs
                        )

                    if not math.isfinite(windowloss):
                        raise FloatingPointError("DPO loss is not finite.")

                    scaler.unscale_(optimizer)

                    gradientnorm = torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=1.0,
                    )

                    gradientvalue = float(gradientnorm.item())

                    if not math.isfinite(gradientvalue):
                        gradientvalue = None

                    previousscale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    updatesucceeded = scaler.get_scale() >= previousscale
                    attemptedsteps += 1

                    if updatesucceeded:
                        optimizersteps += 1
                        consecutiveskips = 0
                    else:
                        skippedsteps += 1
                        consecutiveskips += 1

                    appendpreferencemetrics(
                        metricspath,
                        {
                            "event": "training",
                            "epoch": epoch + 1,
                            "optimizer_step": optimizersteps,
                            "attempted_step": attemptedsteps,
                            "dpo_loss": windowloss,
                            "pairs": windowpairs,
                            "update_skipped": not updatesucceeded,
                            "gradient_scale_before": previousscale,
                            "gradient_scale_after": scaler.get_scale(),
                            "gradient_norm_before_clipping": gradientvalue,
                            "elapsed_seconds": (
                                time.perf_counter() - starttime
                            ),
                        },
                    )

                    if consecutiveskips >= 10:
                        raise FloatingPointError(
                            "Ten consecutive AMP updates were skipped."
                        )

                    progress.update(1)
                    progress.set_postfix(
                        loss=f"{windowloss:.4f}",
                        updates=optimizersteps,
                    )

                    shouldvalidate = (
                        updatesucceeded
                        and optimizersteps % evaluationinterval == 0
                    ) or attemptedsteps == plannedsteps

                    if shouldvalidate:
                        validationresults = evaluatepreferences(
                            model,
                            validationloader,
                            beta,
                        )

                        appendpreferencemetrics(
                            metricspath,
                            {
                                "event": "validation",
                                "optimizer_step": optimizersteps,
                                **validationresults,
                                "elapsed_seconds": (
                                    time.perf_counter() - starttime
                                ),
                            },
                        )

                        if (
                            validationresults["reward_accuracy"]
                            > bestaccuracy
                        ):
                            bestaccuracy = (
                                validationresults["reward_accuracy"]
                            )
                            beststep = optimizersteps

                            savepreferencecheckpoint(
                                model,
                                tokenizer,
                                outputpath / "best",
                                {
                                    "optimizer_step": beststep,
                                    "validation": validationresults,
                                },
                            )

                if attemptedsteps >= plannedsteps:
                    break

        savepreferencecheckpoint(
            model,
            tokenizer,
            outputpath / "final",
            {
                "optimizer_step": optimizersteps,
                "validation": validationresults,
            },
        )

        summary = {
            "optimizer_steps": optimizersteps,
            "attempted_steps": attemptedsteps,
            "skipped_steps": skippedsteps,
            "best_optimizer_step": beststep,
            "best_validation_reward_accuracy": bestaccuracy,
            "final_validation": validationresults,
            "final_gradient_scale": scaler.get_scale(),
            "elapsed_seconds": time.perf_counter() - starttime,
            "peak_allocated_gpu_gib": (
                torch.cuda.max_memory_allocated(device) / 1024 ** 3
            ),
        }

        (outputpath / "summary.json").write_text(
            json.dumps(summary, indent=2, allow_nan=False),
            encoding="utf-8",
        )

        print(json.dumps(summary, indent=2), flush=True)

        return summary

    finally:
        del optimizer
        del model
        gc.collect()
        torch.cuda.empty_cache()

In [14]:
import json
from pathlib import Path

from datasets import DatasetDict, load_from_disk


def buildintactcache(sourcepath, outputpath, minimumtrainpairs=9000):
    sourcepath = Path(sourcepath)
    outputpath = Path(outputpath)

    if outputpath.exists():
        raise FileExistsError(
            f"{outputpath} already exists. Choose a new output path."
        )

    sourcemanifest = json.loads(
        (sourcepath / "manifest.json").read_text(encoding="utf-8")
    )
    eostokenid = sourcemanifest["preparation"]["eos_token_id"]
    cacheddataset = load_from_disk(str(sourcepath / "dataset"))

    def keepintactpair(chosen, rejected):
        return (
            chosen["input_ids"][-1] == eostokenid
            and rejected["input_ids"][-1] == eostokenid
        )

    filtereddatasets = {}
    retained = {}

    for splitname in ("train", "validation"):
        filtereddatasets[splitname] = cacheddataset[splitname].filter(
            keepintactpair,
            input_columns=["chosen", "rejected"],
            keep_in_memory=True,
        )
        retained[splitname] = {
            "before": len(cacheddataset[splitname]),
            "after": len(filtereddatasets[splitname]),
        }

    print(json.dumps(retained, indent=2))

    if retained["train"]["after"] < minimumtrainpairs:
        raise ValueError(
            f"Only {retained['train']['after']} training pairs keep EOS "
            f"after truncation, below {minimumtrainpairs}. Re-prepare the "
            "data with the filter inside preparepreferencesplit instead of "
            "training on a much smaller set."
        )

    if retained["validation"]["after"] < 100:
        raise ValueError(
            "Fewer than 100 validation pairs remain, so reward accuracy "
            "would be too noisy for checkpoint selection."
        )

    outputpath.mkdir(parents=True, exist_ok=False)

    DatasetDict(filtereddatasets).save_to_disk(str(outputpath / "dataset"))

    sourcemanifest["pair_filter"] = (
        "both responses end with EOS after truncation"
    )
    sourcemanifest["pair_filter_counts"] = retained

    (outputpath / "manifest.json").write_text(
        json.dumps(sourcemanifest, indent=2),
        encoding="utf-8",
    )

    return retained


sourcecachepath = next(
    Path("/kaggle/input").rglob("reference_ultrafeedback_full")
)

print("Source cache:", sourcecachepath)

buildintactcache(
    str(sourcecachepath),
    "outputs/reference_ultrafeedback_intact",
)

StopIteration: 

In [18]:
from pathlib import Path

matchingpaths = []

for rootpath in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    for candidatepath in rootpath.rglob("reference_ultrafeedback_full"):
        if candidatepath.is_dir():
            matchingpaths.append(candidatepath)
            print(f"Reference cache folder: {candidatepath}")

            for filepath in sorted(candidatepath.iterdir()):
                print(f"  {filepath.name}")

if not matchingpaths:
    print(
        "The reference cache folder was not found. "
        "Attach the saved DPO training notebook outputs, "
        "or check your locally downloaded outputs folder."
    )

The reference cache folder was not found. Attach the saved DPO training notebook outputs, or check your locally downloaded outputs folder.


In [ ]:
import json
import shutil
from pathlib import Path

runpath = Path("outputs/dpo_ultrafeedback_intact")

print((runpath / "summary.json").read_text(encoding="utf-8"))
print((runpath / "best" / "checkpoint.json").read_text(encoding="utf-8"))

diskusage = shutil.disk_usage("/kaggle/working")
print(f"Free disk space: {diskusage.free / 1024 ** 3:.2f} GiB")

In [19]:
from pathlib import Path

matchingpaths = []

for rootpath in [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]:
    if not rootpath.exists():
        continue

    for candidatepath in rootpath.rglob("*"):
        if not candidatepath.is_dir():
            continue

        foldername = candidatepath.name.lower()

        if (
            "ultrafeedback" in foldername
            and (
                "reference" in foldername
                or "preferences" in foldername
            )
        ):
            matchingpaths.append(candidatepath)

if matchingpaths:
    for candidatepath in sorted(matchingpaths):
        print(f"\nFound folder: {candidatepath}")

        for childpath in sorted(candidatepath.iterdir()):
            print(f"  {childpath.name}")
else:
    print(
        "No UltraFeedback preference or reference-cache folder "
        "was found in the attached inputs or current working directory. "
        "The prepared dataset must be attached or recreated before "
        "we can measure response truncation."
    )


Found folder: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/preferences_ultrafeedback_full
  dataset
  manifest.json

Found folder: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/preferences_ultrafeedback_initial
  dataset
  manifest.json

Found folder: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_ultrafeedback_full
  dataset
  manifest.json

Found folder: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_ultrafeedback_initial
  dataset
  manifest.json


In [20]:
from pathlib import Path

sourcecachepath = Path(
    "/kaggle/input/notebooks/kangerz22/pa2-dpotraining/"
    "pa2-small-model-alignment/outputs/reference_ultrafeedback_full"
)

if not (sourcecachepath / "dataset").is_dir():
    raise FileNotFoundError(
        f"The cached dataset was not found inside {sourcecachepath}."
    )

if not (sourcecachepath / "manifest.json").is_file():
    raise FileNotFoundError(
        f"The reference-cache manifest was not found inside {sourcecachepath}."
    )

print(f"Using reference cache: {sourcecachepath}")

Using reference cache: /kaggle/input/notebooks/kangerz22/pa2-dpotraining/pa2-small-model-alignment/outputs/reference_ultrafeedback_full


In [21]:
import json

from datasets import load_from_disk

manifest = json.loads(
    (sourcecachepath / "manifest.json").read_text(encoding="utf-8")
)

print("Manifest:")
print(json.dumps(manifest, indent=2))

cacheddataset = load_from_disk(
    str(sourcecachepath / "dataset")
)

for splitname, splitdataset in cacheddataset.items():
    print(f"\nSplit: {splitname}")
    print(f"Pairs: {len(splitdataset)}")
    print(f"Columns: {splitdataset.column_names}")

    if len(splitdataset) == 0:
        continue

    firstrecord = splitdataset[0]

    for columnname, columnvalue in firstrecord.items():
        if isinstance(columnvalue, dict):
            print(
                f"{columnname} fields: "
                f"{list(columnvalue.keys())}"
            )
        elif isinstance(columnvalue, (int, float, str, bool)):
            print(f"{columnname}: {columnvalue}")

Manifest:
{
  "preparation": {
    "track": "ultrafeedback",
    "dataset_name": "HuggingFaceH4/ultrafeedback_binarized",
    "dataset_revision": "3949bf5f8c17c394422ccfab0c31ea9c20bdeb85",
    "sft_checkpoint": "/kaggle/input/datasets/kangerz22/pa2sftcheckpoint",
    "checkpoint_metadata": {
      "optimizer_step": 938,
      "validation_loss": 1.3795008834900906,
      "selection": "lowest validation loss"
    },
    "tokenizer_file_sha256": "9c5ae00e602b8860cbd784ba82a8aa14e8feecec692e7076590d014d7b7fdafa",
    "seed": 42,
    "shuffle_buffer": 1000,
    "maximum_sequence_length": 512,
    "maximum_prompt_length": 256,
    "long_prompt_policy": "reject; do not truncate the instruction separately",
    "conversation_policy": "single user and assistant exchange only",
    "prompt_template": "### Instruction:\n{instruction}\n\n### Response:\n",
    "padding_token_id": 151643,
    "eos_token_id": 151643,
    "training": {
      "source_split": "train_prefs",
      "examined_pairs": 1847

In [22]:
import json

eostokenid = manifest["preparation"]["eos_token_id"]
maximumlength = manifest["preparation"]["maximum_sequence_length"]

truncationstatistics = []

for splitname, splitdataset in cacheddataset.items():
    chosenonly = 0
    rejectedonly = 0
    bothmissing = 0
    bothintact = 0
    chosenlengthtotal = 0
    rejectedlengthtotal = 0

    for rownumber, record in enumerate(splitdataset):
        chosenbatch = record["chosen"]
        rejectedbatch = record["rejected"]

        chosenids = chosenbatch["input_ids"]
        rejectedids = rejectedbatch["input_ids"]

        if not chosenids or not rejectedids:
            raise ValueError(
                f"Split {splitname}, row {rownumber} contains "
                "an empty chosen or rejected token sequence."
            )

        for responsename, responsebatch in [
            ("chosen", chosenbatch),
            ("rejected", rejectedbatch),
        ]:
            tokenids = responsebatch["input_ids"]
            labels = responsebatch["labels"]

            if len(tokenids) != len(labels):
                raise ValueError(
                    f"Split {splitname}, row {rownumber}, "
                    f"{responsename} has different token and label lengths."
                )

            if len(tokenids) > maximumlength:
                raise ValueError(
                    f"Split {splitname}, row {rownumber}, "
                    f"{responsename} exceeds the recorded sequence limit."
                )

        chosenmissing = chosenids[-1] != eostokenid
        rejectedmissing = rejectedids[-1] != eostokenid

        if chosenmissing and len(chosenids) < maximumlength:
            raise ValueError(
                f"Split {splitname}, row {rownumber}: chosen "
                "does not end with EOS despite being below the sequence "
                "limit. Inspect preprocessing before assuming truncation."
            )

        if rejectedmissing and len(rejectedids) < maximumlength:
            raise ValueError(
                f"Split {splitname}, row {rownumber}: rejected "
                "does not end with EOS despite being below the sequence "
                "limit. Inspect preprocessing before assuming truncation."
            )

        if chosenmissing and rejectedmissing:
            bothmissing += 1
        elif chosenmissing:
            chosenonly += 1
        elif rejectedmissing:
            rejectedonly += 1
        else:
            bothintact += 1

        chosenlengthtotal += sum(
            label != -100
            for label in chosenbatch["labels"][1:]
        )

        rejectedlengthtotal += sum(
            label != -100
            for label in rejectedbatch["labels"][1:]
        )

    totalpairs = len(splitdataset)

    if totalpairs == 0:
        raise ValueError(
            f"Split {splitname} contains no preference pairs."
        )

    statistics = {
        "split": splitname,
        "total_pairs": totalpairs,
        "chosen_only_missing_eos": chosenonly,
        "rejected_only_missing_eos": rejectedonly,
        "both_missing_eos": bothmissing,
        "both_ending_with_eos": bothintact,
        "chosen_missing_eos_percent": (
            100 * (chosenonly + bothmissing) / totalpairs
        ),
        "rejected_missing_eos_percent": (
            100 * (rejectedonly + bothmissing) / totalpairs
        ),
        "both_ending_with_eos_percent": (
            100 * bothintact / totalpairs
        ),
        "mean_retained_chosen_response_tokens": (
            chosenlengthtotal / totalpairs
        ),
        "mean_retained_rejected_response_tokens": (
            rejectedlengthtotal / totalpairs
        ),
    }

    truncationstatistics.append(statistics)
    print(json.dumps(statistics, indent=2))

{
  "split": "train",
  "total_pairs": 15000,
  "chosen_only_missing_eos": 1934,
  "rejected_only_missing_eos": 943,
  "both_missing_eos": 1947,
  "both_ending_with_eos": 10176,
  "chosen_missing_eos_percent": 25.873333333333335,
  "rejected_missing_eos_percent": 19.266666666666666,
  "both_ending_with_eos_percent": 67.84,
  "mean_retained_chosen_response_tokens": 244.27633333333333,
  "mean_retained_rejected_response_tokens": 216.9762
}
{
  "split": "validation",
  "total_pairs": 1000,
  "chosen_only_missing_eos": 152,
  "rejected_only_missing_eos": 56,
  "both_missing_eos": 128,
  "both_ending_with_eos": 664,
  "chosen_missing_eos_percent": 28.0,
  "rejected_missing_eos_percent": 18.4,
  "both_ending_with_eos_percent": 66.4,
  "mean_retained_chosen_response_tokens": 249.288,
  "mean_retained_rejected_response_tokens": 216.984
}


In [23]:
import copy
import json
import math

from pathlib import Path
from datasets import DatasetDict, load_from_disk

sourcecachepath = Path(
    "/kaggle/input/notebooks/kangerz22/pa2-dpotraining/"
    "pa2-small-model-alignment/outputs/"
    "reference_ultrafeedback_full"
)

outputcachepath = Path(
    "/kaggle/working/pa2-small-model-alignment/outputs/"
    "reference_ultrafeedback_intact"
)

if outputcachepath.exists():
    raise FileExistsError(
        f"The destination already exists: {outputcachepath}. "
        "Use the existing filtered cache if this cell already completed, "
        "or choose a new destination."
    )

sourcemanifest = json.loads(
    (sourcecachepath / "manifest.json").read_text(
        encoding="utf-8"
    )
)

eostokenid = sourcemanifest["preparation"]["eos_token_id"]

sourcedataset = load_from_disk(
    str(sourcecachepath / "dataset")
)

filtereddatasets = {}
filterstatistics = {}
retainedindices = {}

for splitname in ["train", "validation"]:
    splitdataset = sourcedataset[splitname]
    selectedindices = []

    for rownumber, record in enumerate(splitdataset):
        chosenids = record["chosen"]["input_ids"]
        rejectedids = record["rejected"]["input_ids"]

        if not chosenids or not rejectedids:
            raise ValueError(
                f"Split {splitname}, row {rownumber} contains "
                "an empty response sequence."
            )

        if (
            chosenids[-1] != eostokenid
            or rejectedids[-1] != eostokenid
        ):
            continue

        for responsename in ["chosen", "rejected"]:
            responsebatch = record[responsename]

            if responsebatch["labels"][-1] != eostokenid:
                raise ValueError(
                    f"Split {splitname}, row {rownumber}, "
                    f"{responsename}: the final EOS is not supervised."
                )

        for scorename in [
            "reference_chosen_logps",
            "reference_rejected_logps",
        ]:
            if not math.isfinite(float(record[scorename])):
                raise ValueError(
                    f"Split {splitname}, row {rownumber} contains "
                    f"a nonfinite cached score in {scorename}."
                )

        selectedindices.append(rownumber)

    if not selectedindices:
        raise ValueError(
            f"No complete response pairs remain in split {splitname}."
        )

    filtereddataset = splitdataset.select(selectedindices)

    for filterednumber, sourcenumber in enumerate(selectedindices):
        originalrecord = splitdataset[sourcenumber]
        filteredrecord = filtereddataset[filterednumber]

        for columnname in [
            "instruction_hash",
            "chosen",
            "rejected",
            "reference_chosen_logps",
            "reference_rejected_logps",
        ]:
            if originalrecord[columnname] != filteredrecord[columnname]:
                raise ValueError(
                    f"Filtering changed {columnname} in split "
                    f"{splitname}, source row {sourcenumber}."
                )

    filtereddatasets[splitname] = filtereddataset
    retainedindices[splitname] = selectedindices

    filterstatistics[splitname] = {
        "original_pairs": len(splitdataset),
        "retained_pairs": len(filtereddataset),
        "removed_pairs": (
            len(splitdataset) - len(filtereddataset)
        ),
    }

expectedcounts = {
    "train": 10176,
    "validation": 664,
}

for splitname, expectedcount in expectedcounts.items():
    actualcount = len(filtereddatasets[splitname])

    if actualcount != expectedcount:
        raise ValueError(
            f"Split {splitname} retained {actualcount} pairs, "
            f"but the measured dataset should retain {expectedcount}. "
            "Confirm that the original full reference cache is loaded."
        )

outputmanifest = copy.deepcopy(sourcemanifest)

outputmanifest["pair_filter"] = {
    "rule": "Both sequences end with a supervised EOS token",
    "source_cache": str(sourcecachepath),
    "counts": filterstatistics,
    "reference_scores_reused": True,
    "token_sequences_modified": False,
    "original_preparation_metadata_preserved": True,
}

outputcachepath.mkdir(parents=True, exist_ok=False)

DatasetDict(filtereddatasets).save_to_disk(
    str(outputcachepath / "dataset")
)

(outputcachepath / "manifest.json").write_text(
    json.dumps(
        outputmanifest,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

(outputcachepath / "retained_indices.json").write_text(
    json.dumps(retainedindices, indent=2),
    encoding="utf-8",
)

print(json.dumps(filterstatistics, indent=2))
print(f"\nFiltered reference cache saved: {outputcachepath}")
print("Chosen and rejected token sequences were preserved.")
print("Their corresponding cached reference scores were preserved.")
print("Training pairs: 10176")
print("Validation pairs: 664")
print("Expected updates for one epoch at effective batch 32: 318")

Saving the dataset (0/1 shards):   0%|          | 0/10176 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/664 [00:00<?, ? examples/s]

{
  "train": {
    "original_pairs": 15000,
    "retained_pairs": 10176,
    "removed_pairs": 4824
  },
  "validation": {
    "original_pairs": 1000,
    "retained_pairs": 664,
    "removed_pairs": 336
  }
}

Filtered reference cache saved: /kaggle/working/pa2-small-model-alignment/outputs/reference_ultrafeedback_intact
Chosen and rejected token sequences were preserved.
Their corresponding cached reference scores were preserved.
Training pairs: 10176
Validation pairs: 664
Expected updates for one epoch at effective batch 32: 318
